In [1]:
import pandas as pd
import os
import glob

# Carpeta de trabajo
carpeta = r"M:\material_isa\estadisticos\_Turberas"

# Buscar todos los CSV originales
archivos = glob.glob(os.path.join(carpeta, "*.csv"))
archivos = [f for f in archivos if not f.endswith("_periodo_nieve.csv")]

for ruta_entrada in archivos:
    nombre = os.path.basename(ruta_entrada)
    nombre_salida = nombre.replace('.csv', '_periodo_nieve.csv')
    ruta_salida = os.path.join(carpeta, nombre_salida)

    try:
        df = pd.read_csv(ruta_entrada, sep=';', encoding='latin1')
        df.columns = df.columns.str.strip()

        # Asegurar formato de fecha
        df['FECHA'] = pd.to_datetime(df['FECHA'], dayfirst=True, errors='coerce')
        df = df.sort_values('FECHA')
        df['hay_nieve'] = df['NDSI'] > 0.4

        # Cálculo de días sin nieve
        dias_sin_nieve = []
        ultima_fecha_nieve = None

        for _, row in df.iterrows():
            fecha_actual = row['FECHA']
            if pd.isna(fecha_actual):
                dias_sin_nieve.append(None)
                continue

            if row['hay_nieve']:
                ultima_fecha_nieve = fecha_actual
                dias_sin_nieve.append(0)
            elif ultima_fecha_nieve is not None:
                dias_sin_nieve.append((fecha_actual - ultima_fecha_nieve).days)
            else:
                dias_sin_nieve.append(None)

        df['días_sin_nieve'] = dias_sin_nieve

        # Guardar CSV con columnas nuevas
        df.to_csv(ruta_salida, sep=';', index=False, encoding='latin1')
        print(f"✅ Procesado correctamente: {nombre}")

        # ─── ✅ CALCULAR SNOW IN / OFF POR TEMPORADA ─────────────────────────
        df = df.sort_values('FECHA')
        df['Año_hidrologico'] = df['FECHA'].apply(lambda d: d.year if d.month >= 9 else d.year - 1)

        temporadas = df['Año_hidrologico'].unique()
        resumen_temporadas = []

        for temporada in temporadas:
            datos = df[df['Año_hidrologico'] == temporada]
            datos_con_nieve = datos[datos['hay_nieve']]

            if not datos_con_nieve.empty:
                fecha_inicio = datos_con_nieve['FECHA'].min()
                fecha_fin = datos_con_nieve['FECHA'].max()
                resumen_temporadas.append((temporada, fecha_inicio, fecha_fin))

        # Mostrar resultado
        if resumen_temporadas:
            print(f"📅 {nombre_salida} — Periodos de nieve por temporada:")
            for t, ini, fin in resumen_temporadas:
                print(f"   🧊 Temporada {t}-{t+1} ➤ Snow in: {ini.strftime('%d/%m/%Y')} | Snow off: {fin.strftime('%d/%m/%Y')}")
        else:
            print(f"⚠️  {nombre_salida} — No se detectaron períodos de nieve.")

            
    except Exception as e:
        print(f"❌ Error en {nombre}: {e}")


✅ Procesado correctamente: Bassa_Nera.csv
📅 Bassa_Nera_periodo_nieve.csv — Periodos de nieve por temporada:
   🧊 Temporada 2015-2016 ➤ Snow in: 03/12/2015 | Snow off: 01/05/2016
   🧊 Temporada 2016-2017 ➤ Snow in: 07/11/2016 | Snow off: 11/04/2017
   🧊 Temporada 2017-2018 ➤ Snow in: 07/11/2017 | Snow off: 26/04/2018
   🧊 Temporada 2018-2019 ➤ Snow in: 07/11/2018 | Snow off: 27/03/2019
   🧊 Temporada 2019-2020 ➤ Snow in: 27/11/2019 | Snow off: 21/03/2020
   🧊 Temporada 2020-2021 ➤ Snow in: 27/09/2020 | Snow off: 01/03/2021
   🧊 Temporada 2021-2022 ➤ Snow in: 16/12/2021 | Snow off: 15/04/2022
   🧊 Temporada 2022-2023 ➤ Snow in: 06/11/2022 | Snow off: 16/03/2023
   🧊 Temporada 2023-2024 ➤ Snow in: 26/11/2023 | Snow off: 04/04/2024
   🧊 Temporada 2024-2025 ➤ Snow in: 15/12/2024 | Snow off: 16/04/2025
✅ Procesado correctamente: Clots_de_Rialba.csv
📅 Clots_de_Rialba_periodo_nieve.csv — Periodos de nieve por temporada:
   🧊 Temporada 2015-2016 ➤ Snow in: 03/12/2015 | Snow off: 01/05/2016
   🧊